# Load dataset

**record name , balanced sample**
- record 04015 -> 400
- record 04043 -> 1000
- record 04048 -> 900
- record 04908 -> tidak bisa di split karena hanya AF
- record 04936 -> 2000
- record 05091 -> 1000, class 0 29
- record 05121 -> 1000, class 0, 150
- record 05261 -> 1000, class 0 105
- record 06426 -> 2000
- record 06453 -> 300
- record 06995 -> 900
- record 07910 -> 320
- record 08215 -> 400
- record 08219 -> 5000
- record 08378 -> 220
- record 08455 -> 90

In [ ]:
import os

dataset_folder = 'dataset/'
filenames = []
for filename in os.listdir(dataset_folder):
    if filename.find('sequence') > -1 :
        filenames.append(filename)

- Change this parameter

In [ ]:
record = "08219"
n_samples = 5000

In [ ]:
filenames

In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

- Split Dataset

In [ ]:
ecg_df = pd.read_csv(dataset_folder + 'AFDB_%s_sequence_300_pt_2_ch.csv' % record, header=None)

In [ ]:
ecg_df.head()

In [ ]:
X = ecg_df.iloc[:,:600].values
    
y = ecg_df.iloc[:,600].values

le = LabelEncoder()
le.fit(y)

labels = le.classes_
print(" categorical label : ", labels)

y = le.transform(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
                                    X, y, test_size=0.33, random_state=42)

In [ ]:
train_df = pd.DataFrame(np.hstack((X_train, np.expand_dims(y_train, 1))))
test_df = pd.DataFrame(np.hstack((X_test, np.expand_dims(y_test, 1))))

- 1 sequence terdiri dari 600 titik data , kolom terakhir merupakan label untuk tiap sequence

In [ ]:
train_df.head()

## Balancing Dataset
Untuk kebutuhan training model, kita akan atur agar sample tiap class memiliki proporsi yang sama.

In [ ]:
#print dataset distribution

train_df[600]=train_df[600].astype(int)
equilibre=train_df[600].value_counts()

print(equilibre)

- Terlihat jika tiap katerogi memiliki jumlah sample yang berbeda.

In [ ]:
plt.figure(figsize=(20,10))

my_circle = plt.Circle( (0,0), 0.7, color='white')
plt.pie(equilibre, 
        labels=labels, 
        colors=['red','green'],
        autopct='%1.1f%%')

p = plt.gcf()
p.gca().add_artist(my_circle)
plt.show()

- kita akan membuat seluruh class memiliki jumlah sample sebanyak 20000. 
- Sehingga untuk class `0` yang memiliki sample 72471, akan dilakukan pemilihan acak sebanyak 20000 dengan menggunakan Pandas Dataframe `.sample()`.
- Untuk class lainya, karena dibawah 20000 sample, akan diterapkan proses upsampling menjadi 20000 sample menggunakan sklearn `resample()`.

In [ ]:
# sampling and resampling dataset

from sklearn.utils import resample
#n_samples = 1000 
random_states = [42, 123]

dfs = []

for i in range(len(equilibre)):
    dfs.append(train_df[train_df[600]==i])
    if(equilibre[i] > n_samples) :
        dfs[i]=dfs[i].sample(n=n_samples ,random_state=random_states[i])
    else :
        dfs[i]=resample(dfs[i],replace=True,n_samples=n_samples,random_state=random_states[i])

train_df=pd.concat(dfs)

In [ ]:
train_df.head()

Sekarang kita memiliki dataset yang sudah balance.

In [ ]:
# cek balanced dataset

equilibre=train_df[600].value_counts()

print(equilibre)

## Understanding ECG Signal
- Sekarang kita ambil 1 sample tiap class untuk melihat profile heart beat masing-masing class.
- Ini berguna untuk kita agar dapat memahami karakteristik sinyal ECG untuk tiap class.
- Task ini mudah dilakukan menggunakan pandas,  `.groupby()` pada kolom 187 (kolom class) dan dengan menggunakan fungsi `.apply()` kita ambil 1 sample data menggunakan `.sample()`.

In [ ]:
ECG_PER_CLASS =train_df.groupby(600,group_keys=False).apply(lambda train_df : train_df.sample(1))

In [ ]:
ECG_PER_CLASS

- Pertama kita plot tiap sample pada dataframe `ECG_PER_CLASS` menggunakan pandas `.plot()`.
- Setelah itu kita plot 2D histogram untuk keseluruhan sample ECG tiap class menggunakan fungsi `plot_hist()` yang kita buat dibawah.
- Pada plot histogram, daerah dengan intensitas lebih terang menandakan banyak titik sample pada waktu dan nilai ECG tersebut.
- Sebaliknya untuk daerah yang lebih gelap.

In [ ]:
def plot_1_sample(class_number, fs=250):
    data1 = ECG_PER_CLASS.iloc[class_number,300:600]
    data2 = ECG_PER_CLASS.iloc[class_number,0:300]
    times = np.arange(data1.shape[0], dtype='float') / fs * 1000
    
    plt.plot(times, data1)
    plt.plot(times, data2)
    plt.title("Plot 1 sample in class " + str(class_number))
    plt.xlabel("Time (ms)")
    plt.ylabel("Normalized Value")
    plt.legend(['channel 1', 'channel 2'])
    plt.show()

In [ ]:
def plot_hist(class_number, min_, size, fs=250):
    # filter data berdasarkan class_number 
    ECG=train_df.loc[train_df[600]==class_number].values
    
    # scale tm value (*1000) agar bisa ditampilkan dalam satuam (ms)
    tm = np.arange(ECG.shape[1], dtype='float') / fs * 1000
    
    # ambil data hanya pada kolom `min_` sampai `size`
    # convert ke 1D array
    ECG=ECG[:,min_:size]
    ECG_flatten=ECG.flatten()
    
    # generate time data untuk tiap kolom sample, 
    # ulang sebanyak ECG.shape[0]
    times = np.tile(tm[min_:size], ECG.shape[0]) 

    # plot sebagai 2D histogram
    plt.hist2d(times, ECG_flatten, bins=(80,80), cmap=plt.cm.jet)
    plt.title("2D histogram on class " + str(class_number))
    plt.xlabel("Time (ms)")
    plt.ylabel("Normalized Value")
    
    plt.show()

In [ ]:
plot_1_sample(class_number=0)

In [ ]:
plot_hist(class_number=0, min_=0, size=80)

In [ ]:
plot_1_sample(class_number=1)

In [ ]:
plot_hist(class_number=1, min_=0, size=80)

### Save Balanced Training Data and Test Data

In [ ]:
train_df.to_csv(dataset_folder + "train_AFDB_%s_balanced.csv" % record, header=None, index=None)
test_df.to_csv(dataset_folder + "test_AFDB_%s.csv" % record, header=None, index=None)